# ***线性回归的简洁实现***

> 本章使用 PyTorch 的高级 API（`nn.Module`、`nn.MSELoss`、`optim.SGD` 等）以更简洁的方式复现线性回归训练流程，与前一章手动实现形成对比。通过本章，你将掌握 PyTorch 模型搭建、数据加载、训练循环的标准写法。

## 1.1 生成数据集

与手动实现相同，我们使用同样的 `synthetic_data` 函数生成人造线性数据集。

In [46]:
import numpy as np
import torch
import random
from torch.utils import data

In [47]:
def synthetic_data(w, b, num_examples):
    """生成y = wX + b + 噪声"""
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

In [48]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features , labels = synthetic_data(true_w, true_b, 1000)

## 1.2 读取数据集

使用 `TensorDataset` 将特征和标签打包成一个数据集对象，再通过 `DataLoader` 按小批量迭代获取数据，这是 PyTorch 中处理数据的标准范式。

In [49]:
def load_array(data_arrays, batch_size, is_train=True):
    """构造一个Pytorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

In [50]:
batch_size = 10
data_iter = load_array((features, labels), batch_size)

In [51]:
next(iter(data_iter))

[tensor([[ 2.1612,  0.9200],
         [-1.0527, -1.0212],
         [ 0.9857, -1.1890],
         [ 0.0491, -0.4487],
         [ 0.5301,  0.2485],
         [ 0.8443, -0.4699],
         [ 1.7201,  2.3182],
         [ 0.6366, -0.5040],
         [ 0.9596, -1.2229],
         [-1.3136,  0.6118]]),
 tensor([[ 5.3984],
         [ 5.5683],
         [10.2052],
         [ 5.8091],
         [ 4.4010],
         [ 7.4604],
         [-0.2459],
         [ 7.1906],
         [10.2908],
         [-0.5061]])]

## 1.3 定义模型

使用 `nn.Sequential` 容器包裹一个线性层 `nn.Linear(2, 1)`，即构建了一个输入特征为 2、输出为 1 的全连接网络。这是最简单的神经网络定义方式。

In [52]:
from torch import nn
net = nn.Sequential(nn.Linear(2, 1))

## 1.4 初始化模型参数

通过直接操作参数张量的 `data` 属性，将权重初始化为均值为 0、标准差为 0.01 的正态分布，偏置置零。这里未使用复杂的初始化策略，适合简单模型。

In [53]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

## 1.5 定义损失函数

直接调用 PyTorch 内建的 `nn.MSELoss()`，计算预测值与真实值之间的均方误差。损失将自动在反向传播中产生梯度。

In [54]:
loss = nn.MSELoss()

## 1.6 定义优化算法

使用 `torch.optim.SGD` 实例化一个随机梯度下降优化器，传入网络所有参数并设置学习率为 0.03。优化器将自动管理所有参数的更新。

In [55]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

## 1.7 训练

训练循环包含以下标准步骤：
1. **前向传播**：`net(X)` 得到预测输出。
2. **计算损失**：`loss(y_hat, y)`。
3. **梯度清零**：`trainer.zero_grad()`，避免梯度累积。
4. **反向传播**：`l.backward()` 计算梯度。
5. **参数更新**：`trainer.step()` 根据梯度更新参数。

每个 epoch 结束后，用全量数据集计算一次损失，以观察训练进度。

In [56]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.000184
epoch 2, loss 0.000096
epoch 3, loss 0.000095


## 📝 流程总结

本章使用 PyTorch 的高级 API 完成了一个完整的线性回归任务，整体流程可概括为以下七个步骤：

1. **生成数据集** → 调用自定义 `synthetic_data` 产生带有随机噪声的线性数据。
2. **构造数据加载器** → 利用 `TensorDataset` 和 `DataLoader` 实现小批量随机读取。
3. **定义模型** → 通过 `nn.Sequential(nn.Linear(2,1))` 构建单层全连接网络。
4. **初始化参数** → 手动设置权重和偏置的初始值（正态分布 / 零）。
5. **选择损失函数** → 直接使用 `nn.MSELoss()`（均方误差）。
6. **配置优化器** → 实例化 `torch.optim.SGD` 并指定学习率。
7. **训练循环** → 对每个小批量执行：梯度清零 → 前向计算 → 损失计算 → 反向传播 → 参数更新，每轮 epoch 评估整体损失。

与手动实现相比，代码量显著减少，因为 PyTorch 自动处理了梯度计算、参数更新等底层细节。这种模块化风格是后续构建复杂神经网络的基础。